In [1]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import (
    AutoModelForSequenceClassification, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from dotenv import load_dotenv

/home/aryansharma/Desktop/Aryan Pendrive Data/Support-Integrity-Auditor/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

# Small encoder — full fine-tune. ~66M params, fits comfortably in 4GB, no quantization/LoRA needed.
# DistilBERT is a strong, well-supported default for short-text binary classification.
model_name = "distilbert-base-uncased"

print("="*50)
print(f"Loading {model_name} ...")
print("="*50)

tokenizer = AutoTokenizer.from_pretrained(model_name)

num_labels = 4
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=4,
)
model = model.to("cuda")

print(f"Model loaded on: {model.device}")
print(f"Model dtype: {model.dtype}")

Loading distilbert-base-uncased ...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9426.25it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on: cuda:0
Model dtype: torch.float32


In [3]:
# No LoRA / no quantization — DistilBERT is small enough to fully fine-tune on 4GB.
n_total = sum(p.numel() for p in model.parameters())
print(f"Total params: {n_total/1e6:.1f}M (all trainable)")

Total params: 67.0M (all trainable)


In [4]:
training_args = TrainingArguments(
    output_dir="./distilbert_mismatch",
    num_train_epochs=10,                  # encoder fine-tune converges fast; watch eval F1
    per_device_train_batch_size=32,      # DistilBERT is tiny; 4GB handles this easily
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",               # must match save_strategy for load_best_model_at_end
    load_best_model_at_end=True,
    learning_rate=2e-5,                  # standard full fine-tune LR (NOT 2e-4 — that was for LoRA)
    fp16=True,
    report_to="none",
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [5]:
def to_text(row):
    tic_desc = row["Ticket_Description"]
    tic_sub = row["Ticket_Subject"]
    categ = row["Issue_Category"]
    res = row["Resolution_Time_Hours"]
    sc = row["Satisfaction_Score"]

    txt = tic_sub + ": "+ tic_desc + " | " + "Issue Category: " + categ + " | " + "Resolution Time Taken In Hour: " + str(res) + " | "  + "Satisfactory Score: " + str(sc)

    return txt 

In [6]:
df = pd.read_csv("../dataset/preprocessed_data.csv")

In [7]:
df["TXT"] = df.apply(to_text,axis=1)

In [8]:
df.Final_severity_label = df.Final_severity_label.map({"LOW":0,"MEDIUM":1,"HIGH":2, "CRITICAL":3})

In [9]:
dataset = df[["TXT", "Final_severity_label"]]

In [10]:
dataset

,TXT,Final_severity_label
0,"Hours of operation - Individual: Hi Support, W...",0
1,"Data not syncing - Card: Hi Support, The appli...",1
2,"2FA issues - Question: Hi Support, How do I up...",2
3,"Login failed - Let: Hi Support, The dashboard ...",1
4,"Refund status - Attention: Hi Support, I have ...",2
...,...,...
19995,"Installation issue - Think: Hi Support, The ap...",1
19996,"Alert notification - Reality: Hi Support, I re...",3
19997,"Subscription upgrade - Spring: Hi Support, My ...",0
19998,"Suspicious charge - Even: Hi Support, I have b...",2


In [11]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

In [12]:
train_df, eval_df = train_test_split(dataset, test_size=0.2, random_state=42, stratify=dataset["Final_severity_label"])

In [13]:
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

In [14]:
import numpy as np
from sklearn.metrics import accuracy_score,precision_recall_fscore_support

In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = float(accuracy_score(labels, predictions))
    precision, recall, f1_weighted, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    f1_macro = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )[2]
    # Per-class recall (diagnostic — watch the minority CRITICAL class).
    per_class_recall = precision_recall_fscore_support(
        labels, predictions, average=None, zero_division=0
    )[1]

    metrics = {
        "accuracy": accuracy,
        # Macro F1 is the spec metric; expose it as "f1" so
        # metric_for_best_model="f1" resolves (fixes the eval_f1 KeyError).
        "f1": float(f1_macro),
        "f1_macro": float(f1_macro),
        "f1_weighted": float(f1_weighted),
        "precision": float(precision),
        "recall": float(recall),
    }
    for i, r in enumerate(per_class_recall):
        metrics[f"recall_{i}"] = float(r)
    return metrics

In [16]:
def tokenize_function(row):
    return tokenizer(
        row['TXT'],
        truncation=True,
        max_length=128   # tickets are ~55 tokens; 128 is plenty of headroom
    )

In [17]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 4000/4000 [00:00<00:00, 25355.41 examples/s]


In [18]:
tokenized_train = tokenized_train.remove_columns(['TXT'])
tokenized_eval = tokenized_eval.remove_columns(['TXT'])

In [19]:
tokenized_train = tokenized_train.rename_column("Final_severity_label", 'labels')
tokenized_eval = tokenized_eval.rename_column("Final_severity_label", 'labels')

# Set format for PyTorch
tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_eval.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Data collator for dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Train
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,F1 Macro,F1 Weighted,Precision,Recall,Recall 0,Recall 1,Recall 2,Recall 3
1,0.223321,0.233819,0.919750,0.912418,0.912418,0.919726,0.921768,0.919750,0.932606,0.973580,0.868876,0.952862
2,0.178496,0.193245,0.919000,0.914472,0.914472,0.918562,0.923103,0.919000,0.971759,0.928666,0.842219,0.976431
3,0.153452,0.160057,0.938500,0.933637,0.933637,0.938840,0.940724,0.938500,0.927471,0.922061,0.960375,0.936027
4,0.143529,0.135214,0.945250,0.940388,0.940388,0.945340,0.945700,0.945250,0.942875,0.944518,0.953890,0.919192
5,0.138174,0.145583,0.946250,0.940007,0.940007,0.946392,0.946876,0.946250,0.948652,0.960370,0.934438,0.952862
6,0.105395,0.162964,0.948000,0.943001,0.943001,0.948020,0.948235,0.948000,0.951220,0.973580,0.932277,0.939394
7,0.067844,0.192095,0.945250,0.939618,0.939618,0.945408,0.945869,0.945250,0.942234,0.957728,0.943084,0.939394
8,0.072433,0.232292,0.943250,0.939433,0.939433,0.943261,0.943516,0.943250,0.944801,0.968296,0.927954,0.942761
9,0.025189,0.250368,0.942000,0.935348,0.935348,0.942120,0.942419,0.942000,0.942234,0.953765,0.941643,0.912458
10,0.019315,0.259730,0.942500,0.937372,0.937372,0.942579,0.942809,0.942500,0.940308,0.961691,0.939481,0.919192


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.67it/s]


TrainOutput(global_step=5000, training_loss=0.14575453344881534, metrics={'train_runtime': 2280.3583, 'train_samples_per_second': 70.164, 'train_steps_per_second': 2.193, 'total_flos': 2202746467762176.0, 'train_loss': 0.14575453344881534, 'epoch': 10.0})

In [21]:
from sklearn.metrics import classification_report,confusion_matrix

In [23]:
pred = trainer.predict(tokenized_eval)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

print("Eval metrics:", {k: round(v, 4) for k, v in pred.metrics.items() if k.startswith("test_")})
print("\nClassification report:")
print(classification_report(y_true, y_pred, digits=4))
print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(y_true, y_pred))

Eval metrics: {'test_loss': 0.1352, 'test_accuracy': 0.9453, 'test_f1': 0.9404, 'test_f1_macro': 0.9404, 'test_f1_weighted': 0.9453, 'test_precision': 0.9457, 'test_recall': 0.9453, 'test_recall_0': 0.9429, 'test_recall_1': 0.9445, 'test_recall_2': 0.9539, 'test_recall_3': 0.9192, 'test_runtime': 9.9548, 'test_samples_per_second': 401.817, 'test_steps_per_second': 6.329}

Classification report:
              precision    recall  f1-score   support

           0     0.9595    0.9429    0.9511      1558
           1     0.9688    0.9445    0.9565       757
           2     0.9252    0.9539    0.9393      1388
           3     0.9100    0.9192    0.9146       297

    accuracy                         0.9453      4000
   macro avg     0.9409    0.9401    0.9404      4000
weighted avg     0.9457    0.9453    0.9453      4000

Confusion matrix [rows=true, cols=pred]:
[[1469    7   82    0]
 [  12  715   13   17]
 [  44   10 1324   10]
 [   6    6   12  273]]


In [35]:
e = df.iloc[eval_df.index]
e["Priority_Level"] = e["Priority_Level"].map({"Low":0,"Medium":1,"High":2,"Critical":3})
e

,Ticket_ID,Customer_Name,Customer_Email,Ticket_Subject,Ticket_Description,Issue_Category,Priority_Level,Ticket_Channel,Submission_Date,Resolution_Time_Hours,Assigned_Agent,Satisfaction_Score,low_score,medium_score,high_score,critical_score,Final_severity_label,Is_Mismatch,TXT
16121,TKT-116121,Susan Michael,David.Schroeder@enterprise.org,Suspicious charge - Lay,"Hi Support, I requested a refund 5 days ago, w...",Billing,0,Web Form,2024-08-12,24,Ben Carter,5,0.252898,0.168805,0.349181,0.187218,2,1,"Suspicious charge - Lay: Hi Support, I request..."
3465,TKT-103465,Adam Carter,michaelrodriguez@example.org,Payment failed - Pattern,"Hi Support, I have been trying to update my pa...",Billing,1,Web Form,2025-09-28,41,Elena Rodriguez,1,0.254416,0.230283,0.563386,0.195987,2,1,"Payment failed - Pattern: Hi Support, I have b..."
2586,TKT-102586,Lauren Robertson,Darren.Cooper@company.com,Refund status - Me,"Hi Support, Why is my bill higher than the agr...",Billing,0,Chat,2025-03-20,41,Chloe Adams,4,0.255790,0.203411,0.357747,0.111266,2,1,"Refund status - Me: Hi Support, Why is my bill..."
4600,TKT-104600,Jason Smith,Michael.Anderson@enterprise.org,Subscription upgrade - Subject,"Hi Support, I am not receiving the password re...",Account,1,Chat,2024-04-26,15,Elena Rodriguez,4,0.512316,0.184701,0.329429,0.247272,0,1,"Subscription upgrade - Subject: Hi Support, I ..."
11049,TKT-111049,Jim Cox,tsanchez@example.com,Login failed - Congress,"Hi Support, The application crashes every time...",Technical,2,Chat,2024-08-17,1,Chloe Adams,5,0.248284,0.367937,0.342003,0.260695,1,1,"Login failed - Congress: Hi Support, The appli..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9519,TKT-109519,Brittney Solomon,frankjoshua@example.com,Stolen card - Rest,"Hi Support, Someone used my card to make a pur...",Fraud,3,Web Form,2024-11-23,6,Chloe Adams,4,0.179497,0.100821,0.312218,0.393362,3,0,"Stolen card - Rest: Hi Support, Someone used m..."
7671,TKT-107671,Candace Holloway,Michael.Reyes@enterprise.org,Subscription upgrade - List,"Hi Support, My profile picture is not updating...",Account,1,Email,2025-02-07,19,David Kim,4,0.424518,0.203243,0.263981,0.162543,0,1,"Subscription upgrade - List: Hi Support, My pr..."
10566,TKT-110566,Janet Williams,Jonathan.Collins@enterprise.org,Refund status - Cold,"Hi Support, I noticed a double charge on my st...",Billing,1,Email,2025-02-02,37,Anya Sharma,2,0.263795,0.192913,0.363051,0.159238,2,1,"Refund status - Cold: Hi Support, I noticed a ..."
5579,TKT-105579,Sharon Haley,ryan75@example.com,Unrecognized login - Step,"Hi Support, I received a suspicious email clai...",Fraud,3,Chat,2025-10-26,7,Anya Sharma,5,0.332393,0.132572,0.321790,0.365650,3,0,"Unrecognized login - Step: Hi Support, I recei..."


In [36]:
s = (y_pred != e["Priority_Level"]).astype(int)

In [38]:
yt = e["Is_Mismatch"]

In [40]:
print("\nClassification report:")
print(classification_report(yt, s, digits=4))
print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(yt, s))


Classification report:
              precision    recall  f1-score   support

           0     0.9706    0.9341    0.9520      1412
           1     0.9648    0.9845    0.9746      2588

    accuracy                         0.9667      4000
   macro avg     0.9677    0.9593    0.9633      4000
weighted avg     0.9668    0.9667    0.9666      4000

Confusion matrix [rows=true, cols=pred]:
[[1319   93]
 [  40 2548]]


In [4]:
import torch
import gc

# 1. Basic cache clear - releases unused cached memory
torch.cuda.empty_cache()

# 2. Force Python garbage collection
gc.collect()

# 3. Synchronize all CUDA streams
# torch.cuda.synchronize()

0

In [2]:
import torch

# Check current GPU memory state
def check_gpu_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3  # GB
        reserved = torch.cuda.memory_reserved() / 1024**3    # GB
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        
        # Free memory calculation
        free = total - allocated  # Not entirely accurate due to caching
        actually_free = total - reserved  # More accurate for "can I allocate more?"
        
        print(f"Total GPU Memory:     {total:.2f} GB")
        print(f"Allocated by PyTorch: {allocated:.2f} GB")
        print(f"Reserved (cached):    {reserved:.2f} GB")
        print(f"≈ Actually Free:      {actually_free:.2f} GB")
        print(f"PyTorch-reported free:{free:.2f} GB")
    else:
        print("CUDA not available")

check_gpu_memory()

Total GPU Memory:     3.68 GB
Allocated by PyTorch: 0.00 GB
Reserved (cached):    0.00 GB
≈ Actually Free:      3.68 GB
PyTorch-reported free:3.68 GB
